In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [16]:
import os
print(os.getcwd())

C:\Users\practicanteit3\Documents\MBA\notebooks


In [17]:
df = pd.read_csv('../processed/outlier.csv', sep=';')
df_historial = pd.read_csv('../raw/ConsumoMasivoFiltro.csv', sep=';',low_memory=False)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cliente             5 non-null      int64  
 1   frecuencia          5 non-null      int64  
 2   recencia            5 non-null      int64  
 3   ticket_promedio     5 non-null      float64
 4   ticket              5 non-null      float64
 5   referencias_unicas  5 non-null      int64  
 6   margen_promeido     5 non-null      float64
 7   margen              5 non-null      float64
 8   avg_descuento       5 non-null      float64
 9   cl_value            5 non-null      float64
 10  linea_favorita      5 non-null      object 
 11  marca_favorita      5 non-null      object 
dtypes: float64(6), int64(4), object(2)
memory usage: 612.0+ bytes


In [19]:
df.describe()

,cliente,frecuencia,recencia,ticket_promedio,ticket,referencias_unicas,margen_promeido,margen,avg_descuento,cl_value
count,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00
mean,"784,430,907.40",21.80,14.00,"2,824,811.38","98,851,474.61",7.20,"1,747,451.20","61,940,018.43",0.08,"67,321,024.07"
std,"455,069,086.44",7.33,12.19,"2,437,645.34","102,548,257.50",3.42,"1,692,340.84","70,307,430.50",0.07,"74,803,264.28"
min,"3,661,187.00",14.00,2.00,"1,355,991.53","32,041,807.08",3.00,"842,617.00","21,171,112.12",0.00,"24,470,726.82"
25%,"811,047,163.00",17.00,3.00,"1,747,909.06","43,346,810.60",5.00,"882,334.43","22,114,640.40",0.00,"32,041,807.08"
50%,"901,295,332.00",19.00,12.00,"1,884,812.18","59,428,908.00",8.00,"1,005,210.93","28,648,977.98",0.12,"37,435,881.88"
75%,"1,077,426,273.00",28.00,24.00,"1,970,309.57","80,003,500.29",8.00,"1,245,359.54","52,057,731.52",0.12,"42,035,737.44"
max,"1,128,724,582.00",31.00,29.00,"7,165,034.54","279,436,347.08",12.00,"4,761,734.11","185,707,630.13",0.14,"200,620,967.14"


In [20]:
X = df[['frecuencia', 'ticket_promedio','cl_value']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [21]:
df_mba = pd.merge(df_historial[['cliente','nombrecliente','referencia','nombrereferencia','cantidad','factura']], df, on='cliente', how = 'inner')
df_mba.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cliente             171 non-null    int64  
 1   nombrecliente       171 non-null    object 
 2   referencia          171 non-null    object 
 3   nombrereferencia    171 non-null    object 
 4   cantidad            171 non-null    float64
 5   factura             171 non-null    object 
 6   frecuencia          171 non-null    int64  
 7   recencia            171 non-null    int64  
 8   ticket_promedio     171 non-null    float64
 9   ticket              171 non-null    float64
 10  referencias_unicas  171 non-null    int64  
 11  margen_promeido     171 non-null    float64
 12  margen              171 non-null    float64
 13  avg_descuento       171 non-null    float64
 14  cl_value            171 non-null    float64
 15  linea_favorita      171 non-null    object 
 16  marca_fa

In [22]:
basket = df_mba.groupby('factura')['referencia'].apply(list)
indices = basket.index.tolist()

In [23]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori,association_rules

te = TransactionEncoder()
te_ary = te.fit(basket).transform(basket)
df_mba_analysis = pd.DataFrame(te_ary,columns = te.columns_, index = indices )



In [24]:
df_mba_analysis = apriori(df_mba_analysis, min_support= 0.1, use_colnames=True)
df_mba_analysis['productos'] = df_mba_analysis['itemsets'].apply( lambda x: len(x)) 
df_mba_analysis.sort_values(by='support', ascending=False)

,support,itemsets,productos
3,0.61,(TRAALR6ALB2),1
1,0.16,(TRAAALR03ALB2),1
5,0.15,(TRDR20RJBLK180),1
0,0.14,(MT121000S),1
4,0.14,(TRAAR6EHDR1),1
6,0.14,"(TRAALR6ALB2, TRAAALR03ALB2)",2
2,0.11,(TRAAAR03EHDBLK),1
7,0.10,"(TRAAR6EHDR1, TRAAAR03EHDBLK)",2


In [25]:
reglas = association_rules(df_mba_analysis,metric='confidence', min_threshold=0.5)

In [26]:
reglas.sort_values(by='confidence', ascending=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2,(TRAAAR03EHDBLK),(TRAAR6EHDR1),0.11,0.14,0.10,0.92,6.66,1.00,0.09,10.35,0.96,0.69,0.90,0.82
0,(TRAAALR03ALB2),(TRAALR6ALB2),0.16,0.61,0.14,0.88,1.44,1.00,0.04,3.28,0.36,0.22,0.69,0.55
1,(TRAAR6EHDR1),(TRAAAR03EHDBLK),0.14,0.11,0.10,0.73,6.66,1.00,0.09,3.34,0.99,0.69,0.70,0.82


In [27]:
df_mba.to_parquet('../reglas/segmento_outlier/Basket.csv')
reglas.to_csv('../reglas/segmento_outlier/reglas.csv',sep=';')

In [28]:
df_mba.head()

,cliente,nombrecliente,referencia,nombrereferencia,cantidad,factura,frecuencia,recencia,ticket_promedio,ticket,referencias_unicas,margen_promeido,margen,avg_descuento,cl_value,linea_favorita,marca_favorita
0,901295332,DISTRIBUIDORA EL NUEVO REY SAS,ALC013654-12,ALPINO COLORES BORRABLES X12,1.00,HHME001311124,31,2,"1,355,991.53","80,003,500.29",12,"882,334.43","52,057,731.52",0.12,"42,035,737.44",ALKALINAS,TRONEX
1,811047163,ILUMINACION Y ENERGIA S.A.S.,MT121000S,BATERIA SELLADA VRLA AGM MTEK 12V 100A,2.00,HHME001426794,17,24,"1,884,812.18","32,041,807.08",3,"1,245,359.54","21,171,112.12",0.00,"32,041,807.08",OTROS,MTEK
2,811047163,ILUMINACION Y ENERGIA S.A.S.,MT121000S,BATERIA SELLADA VRLA AGM MTEK 12V 100A,2.00,HHME001540992,17,24,"1,884,812.18","32,041,807.08",3,"1,245,359.54","21,171,112.12",0.00,"32,041,807.08",OTROS,MTEK
3,811047163,ILUMINACION Y ENERGIA S.A.S.,MT121000S,BATERIA SELLADA VRLA AGM MTEK 12V 100A,2.00,ME000564071,17,24,"1,884,812.18","32,041,807.08",3,"1,245,359.54","21,171,112.12",0.00,"32,041,807.08",OTROS,MTEK
4,811047163,ILUMINACION Y ENERGIA S.A.S.,MT121000S,BATERIA SELLADA VRLA AGM MTEK 12V 100A,2.00,ME000565814,17,24,"1,884,812.18","32,041,807.08",3,"1,245,359.54","21,171,112.12",0.00,"32,041,807.08",OTROS,MTEK
